In [0]:
movies = spark.table("databricks_ws.silver.movies")

tables = [
    "movie_casts",
    "movie_languages",
    "movie_companies",
    "production_countries"
]

for table in tables:
    df = spark.table(f"databricks_ws.silver.{table}")

    orphans = df.join(
        movies.select("movie_id"), "movie_id", "left_anti"
    ).count()

    print(f"{table}: {orphans} orphan relationships")

movie_casts: 390 orphan relationships
movie_languages: 132 orphan relationships
movie_companies: 209 orphan relationships
production_countries: 112 orphan relationships


In [0]:
from pyspark.sql.functions import col

# Silver validation
movies = spark.table("databricks_ws.silver.movies")
genres = spark.table("databricks_ws.silver.movie_genres")

duplicates = genres.count() - genres.select("movie_id", "genre_id").distinct().count()

orphans = genres.join(
    movies.select("movie_id"), "movie_id", "left_anti"
).count()

print("=== SILVER ===")
print("Total relationships:", genres.count())
print("Duplicates:", duplicates)
print("Orphan relationships:", orphans)

assert duplicates == 0, "Duplicates found in Silver"

# Gold validation
dim_movies = spark.table("databricks_ws.gold.dim_movies")

bridges = [
    ("bridge_movie_genres", "genre_id", "dim_genres"),
    ("bridge_movie_casts", "person_id", "dim_persons"),
    ("bridge_movie_languages", "language_id", "dim_languages"),
    ("bridge_movie_prod_companies", "company_id", "dim_prod_companies"),
    ("bridge_movie_prod_countries", "country_id", "dim_countries")
]

print("\n=== GOLD ===")

errors = []

for name, key, dimension in bridges:
    bridge = spark.table(f"databricks_ws.gold.{name}")
    dim = spark.table(f"databricks_ws.gold.{dimension}")

    missing_movies = bridge.join(
        dim_movies.select("movie_id"), "movie_id", "left_anti"
    ).count()

    missing_refs = bridge.join(
        dim.select(key), key, "left_anti"
    ).count()

    duplicates = bridge.count() - bridge.select("movie_id", key).distinct().count()

    status = "PASS" if missing_movies == 0 and missing_refs == 0 and duplicates == 0 else "FAIL"

    print(f"\n{name}: {status}")
    print("Missing movies:", missing_movies)
    print("Missing references:", missing_refs)
    print("Duplicates:", duplicates)

    if status == "FAIL":
        errors.append(name)

assert not errors, f"Validation failed: {errors}"

print("\nAll Gold validations passed!")


=== SILVER ===
Total relationships: 12160
Duplicates: 0
Orphan relationships: 225

=== GOLD ===

bridge_movie_genres: PASS
Missing movies: 0
Missing references: 0
Duplicates: 0

bridge_movie_casts: PASS
Missing movies: 0
Missing references: 0
Duplicates: 0

bridge_movie_languages: PASS
Missing movies: 0
Missing references: 0
Duplicates: 0

bridge_movie_prod_companies: PASS
Missing movies: 0
Missing references: 0
Duplicates: 0

bridge_movie_prod_countries: PASS
Missing movies: 0
Missing references: 0
Duplicates: 0

All Gold validations passed!
